# Register-family analyzers

This is **stage 1 of the 6-stage producer chain**. It reloads the cache written by stage 00, runs the four register-family per-doc analyzers, and writes a partial JSON keyed by file index.

| Analyzer | What it produces (per-file) |
|---|---|
| `mood_for_doc` | Imperative-marker lexical density (count + `pct` + `per_sent`). |
| `register_for_doc` | TTR, mean sentence length, dependency depth, Heylighen F-score, four register classes. |
| `stance_for_doc` | Five lexical stance classes (`directive` / `expository` / `positive_evaluative` / `negative_evaluative` / `dialogic`) + 1p/2p engagement + the `positive_evaluative` quality/emphasis split. |
| `sentence_register_for_doc` | Per-sentence multi-label classifier (six classes: `collaborative` / `permissive` / `appreciative` / `imperative` / `directive` / `configuring`) + addressee classification for appreciative/collaborative. |

Output: `_pipeline_cache/partial_register.json` (per-file metric trees).

In [1]:
"""Reload corpus + DocBin from stage 00."""
import os, pathlib, json, importlib
import pandas as pd
from tqdm.auto import tqdm
from spacy.tokens import DocBin

_here = pathlib.Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in [_here, *_here.parents] if (p / "prompt_pipeline.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        f"Could not find prompt_pipeline.py walking up from {_here}. "
        "Run from inside the claude-prompts-analysis repo."
    )
if pathlib.Path.cwd() != PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)

CACHE_DIR  = PROJECT_ROOT / "_pipeline_cache"
DOCBIN_IN  = CACHE_DIR / "corpus_docs.spacy"
META_IN    = CACHE_DIR / "corpus_meta.parquet"
PARTIAL_OUT = CACHE_DIR / "partial_register.json"

assert DOCBIN_IN.exists(),  f"missing {DOCBIN_IN} — run 00_setup_and_corpus first"
assert META_IN.exists(),    f"missing {META_IN} — run 00_setup_and_corpus first"

import prompt_pipeline
importlib.reload(prompt_pipeline)
from prompt_pipeline import NLP

df = pd.read_parquet(META_IN)
docs = list(DocBin().from_disk(DOCBIN_IN).get_docs(NLP.vocab))
assert len(docs) == len(df), f"DocBin/df length mismatch: {len(docs)} vs {len(df)}"
print(f"reloaded {len(df)} files, {sum(d.__len__() for d in docs):,} doc tokens")

reloaded 290 files, 145,534 doc tokens


## 1. Mood

**Mood marker density** is the share of word tokens matched by `IMPERATIVE_MARKERS` (the lexicon is echoed verbatim into the YAML's `lexicons` block). Reported per file as count + `pct` (% of word tokens) + `per_sent` (rate per sentence). The per-sentence imperative classifier moved into `sentence_register`; this block carries only the lexical-density signal (`marker_*`).

In [2]:
from prompt_pipeline import mood_for_doc

mood_per_file = [mood_for_doc(d, n, s)
                 for d, n, s in zip(docs, df["n_tokens"], df["n_sents"])]
df_mood = pd.DataFrame(mood_per_file)
print("per-file mood (head):")
print(df_mood.head().to_string())
print()
print("category mean (mood — imperative-marker density):")
print(pd.concat([df[["category"]], df_mood], axis=1)
        .groupby("category").mean(numeric_only=True).round(3).to_string())

per-file mood (head):
   marker_count  marker_pct  marker_per_sent
0             6      0.6410           0.2727
1             0      0.0000           0.0000
2            16      0.4529           0.1391
3             4      1.0667           0.1538
4             1      0.5556           0.2000

category mean (mood — imperative-marker density):
                  marker_count  marker_pct  marker_per_sent
category                                                   
Agent prompt             5.459       0.773            0.224
Data / template          4.667       0.573            0.123
Skill                    8.400       0.603            0.162
System prompt            2.328       1.093            0.264
System reminder          1.700       1.032            0.241
Tool description         2.190       2.136            0.412
Tool parameter           0.000       0.000            0.000


## 2. Register

**Register** captures formality. Four numerical metrics per file:

- **TTR** (type-token ratio) — unique types ÷ total tokens. Anti-correlates with file length (longer files reuse vocabulary, lower TTR).
- **Mean sentence length** — tokens per sentence.
- **Mean dependency depth** — average nesting depth of the spaCy parse tree.
- **Heylighen F-score** (Heylighen & Dewaele 2002) — a 0–100 formality index: `F = 50 + 0.5 × (noun + adj + prep + article − pronoun − verb − adverb − interjection)`, each term as a percentage of all tokens. Higher = more formal-academic prose; the corpus clusters in the 70–80 band.

Plus lexical density for four register classes (`frozen` / `formal` / `consultative` / `casual`).

In [3]:
from prompt_pipeline import register_for_doc

register_per_file = [register_for_doc(d, n, s)
                     for d, n, s in zip(docs, df["n_tokens"], df["n_sents"])]
df_register = pd.DataFrame(register_per_file)
print("per-file register (head):")
print(df_register.head().to_string())
print()
print("category mean (numeric register cols):")
num_cols = [c for c in df_register.columns
            if c != "dominant_register" and not c.endswith("_count")]
print(pd.concat([df[["category"]], df_register[num_cols]], axis=1)
        .groupby("category").mean(numeric_only=True).round(3).to_string())

per-file register (head):
      ttr  mean_sent_len  dep_depth  f_score  frozen_count  formal_count  consultative_count  casual_count  frozen_pct  formal_pct  consultative_pct  casual_pct  frozen_per_sent  formal_per_sent  consultative_per_sent  casual_per_sent dominant_register
0  0.5407          42.55      3.910    71.98             0             0                   5             1         0.0         0.0            0.5342      0.1068              0.0              0.0                 0.2273           0.0455      consultative
1  0.6421          18.62      2.661    75.89             0             0                   0             1         0.0         0.0            0.0000      0.4132              0.0              0.0                 0.0000           0.0769            casual
2  0.3840          30.72      2.924    61.05             0             0                   8            28         0.0         0.0            0.2264      0.7925              0.0              0.0                 0.06

## 3. Stance

**Stance** classifies expressed attitude into five polarity-aware classes: `directive`, `expository`, `positive_evaluative`, `negative_evaluative`, `dialogic`. Each is a hand-curated lexicon match, normalized as `pct` (% of word tokens) and `per_sent`. We also count **1st/2nd-person engagement** (`pronouns_1p`, `pronouns_2p`) since they covary with stance.

**The `positive_evaluative` quality / emphasis split.** The union `positive_evaluative` lexicon conflates two phenomena, so each file also carries the split:

- `positive_evaluative_quality` — genuinely affirmative tone: `good`, `recommended`, `optimal`, `safe`.
- `positive_evaluative_emphasis` — emphasis-of-rule words: `important`, `critical`, `essential`, `key`.

The union is preserved for back-compat in existing charts. When the question is *how much praise is here*, cite the quality-only ratio against `negative_evaluative`. When the question is *how loud is the rule emphasis*, cite the emphasis count alongside the imperative-marker density.

In [4]:
from prompt_pipeline import stance_for_doc

stance_per_file = [stance_for_doc(d, n, s)
                   for d, n, s in zip(docs, df["n_tokens"], df["n_sents"])]
df_stance = pd.DataFrame(stance_per_file)
print("per-file stance (head):")
print(df_stance.head().to_string())
print()
print("dominant stance by category:")
print(pd.crosstab(df["category"], df_stance["dominant_stance"]))
print()
print("category mean stance (% of words and per-sentence rate):")
num_cols = [c for c in df_stance.columns
            if c != "dominant_stance" and not c.endswith("_count")]
print(pd.concat([df[["category"]], df_stance[num_cols]], axis=1)
        .groupby("category").mean(numeric_only=True).round(3).to_string())

per-file stance (head):
   directive_count  directive_pct  directive_per_sent  expository_count  expository_pct  expository_per_sent  positive_evaluative_count  positive_evaluative_pct  positive_evaluative_per_sent  negative_evaluative_count  negative_evaluative_pct  negative_evaluative_per_sent  dialogic_count  dialogic_pct  dialogic_per_sent  pronouns_1p_count  pronouns_1p_pct  pronouns_1p_per_sent  pronouns_2p_count  pronouns_2p_pct  pronouns_2p_per_sent dominant_stance  positive_evaluative_quality_count  positive_evaluative_quality_pct  positive_evaluative_quality_per_sent  positive_evaluative_emphasis_count  positive_evaluative_emphasis_pct  positive_evaluative_emphasis_per_sent
0               18         1.9231              0.8182                15          1.6026               0.6818                          6                   0.6410                        0.2727                          0                   0.0000                        0.0000               3        0.3205     

## 3b. Sentence-level pragmatic register

Per-sentence **multi-label** classifier with six classes: `collaborative` / `permissive` / `appreciative` / `imperative` / `directive` / `configuring`. Implemented via `M_SENT_REGISTER` PhraseMatchers, `M_STANCE["directive"]`, and a spaCy `DependencyMatcher` for parse-tree cues; the `imperative` flag is driven by `classify_sent_mood`. Near-zero classes are deliberately preserved — absence is the welfare-relevant signal for this corpus.

**Multi-label semantics.** A single sentence can carry several flags simultaneously. The hypothetical sentence `"Please, we should consider running the migration."` is `permissive` (`please`), `collaborative` (`we should`), `imperative` (`consider`), and `directive` (modal `should`). Because each contributes to multiple class counts, per-class `sent_pct` values across the six classes can sum to more than 100% within a category — intentional, not a bug.

In [5]:
from prompt_pipeline import sentence_register_for_doc

sentence_register_per_file = [
    sentence_register_for_doc(d, s)
    for d, s in zip(docs, df["n_sents"])
]
df_sent_register = pd.DataFrame(sentence_register_per_file)

print("per-file sentence_register (head):")
print(df_sent_register.head().to_string())
print()
print("dominant by category:")
print(pd.crosstab(df["category"], df_sent_register["dominant"]))
print()
print("category mean sentence_register (% of sentences):")
pct_cols = [c for c in df_sent_register.columns if c.endswith("_sent_pct")]
print(pd.concat([df[["category"]], df_sent_register[pct_cols]], axis=1)
        .groupby("category").mean(numeric_only=True).round(2).to_string())

per-file sentence_register (head):
   collaborative_sent_count  collaborative_sent_pct  permissive_sent_count  permissive_sent_pct  appreciative_sent_count  appreciative_sent_pct  imperative_sent_count  imperative_sent_pct  directive_sent_count  directive_sent_pct  configuring_sent_count  configuring_sent_pct  none_sent_count  none_sent_pct  appreciative_addressee_claude_count  appreciative_addressee_user_count  appreciative_addressee_unknown_count  collaborative_addressee_claude_count  collaborative_addressee_user_count  collaborative_addressee_unknown_count    dominant
0                         0                     0.0                      3              13.6364                        0                    0.0                      7              31.8182                     9             40.9091                       5               22.7273                7        31.8182                                    0                                  0                                     0     

## Write `partial_register.json`

One JSON document, keyed by file index (string), with the four metric blocks per file. Stage 04 reloads this and feeds it directly to `build_file_record`.

In [6]:
partial = {
    str(i): {
        "mood":              mood_per_file[i],
        "register":          register_per_file[i],
        "stance":            stance_per_file[i],
        "sentence_register": sentence_register_per_file[i],
    }
    for i in range(len(df))
}
with open(PARTIAL_OUT, "w") as f:
    json.dump(partial, f)
size = PARTIAL_OUT.stat().st_size
print(f"wrote {PARTIAL_OUT.relative_to(PROJECT_ROOT)}  ({size:,} bytes, {size/1024:.1f} KiB)")
print(f"      {len(partial)} per-file records, 4 blocks each")

wrote _pipeline_cache/partial_register.json  (598,693 bytes, 584.7 KiB)
      290 per-file records, 4 blocks each
